# 01.1. Comparación de encoders

Comparar recuperación sobre el corpus completo con las 50 preguntas oficiales. Se prueban primero los encoders recomendados. La referencia se consulta después de recuperar, solo para medir coincidencias de normas.

Recall de normas no mide corrección jurídica ni garantiza recuperar el artículo adecuado. Las preguntas sin citas extraíbles quedan fuera de esa métrica. Los resultados son de desarrollo, no una evaluación independiente.

## Entorno

El mismo notebook corre en local o en Colab. En Colab, subir ai-week-colab.zip a Mi unidad/AIWEEK y seleccionar GPU A100. El ZIP ya incluye los datos necesarios.

Los datos se extraen al disco de la sesión. Los resultados nuevos se guardan en Drive mientras avanza la ejecución. Los modelos se descargan al disco de Colab. Si se elimina la sesión, se descargan de nuevo.

Si la instalación pide reiniciar, reiniciar la sesión y volver a ejecutar desde esta celda. MODO puede ser auto, local o colab.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import zipfile

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"
try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
if MODO not in {"auto", "local", "colab"}:
    raise ValueError("MODO debe ser auto, local o colab")
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete_drive = DRIVE / "ai-week-colab.zip"
    if not paquete_drive.is_file():
        raise FileNotFoundError(f"Subir ai-week-colab.zip a {DRIVE}")
    paquete = Path("/content/ai-week-colab.zip")
    shutil.copy2(paquete_drive, paquete)
    huella_paquete = hash_archivo(paquete)
    raiz = Path("/content/aiweek") / huella_paquete[:12]
    if not (raiz / ".paquete_verificado").is_file():
        raiz.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(raiz)
        inventario = json.loads((raiz / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(raiz / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (raiz / ".paquete_verificado").write_text(huella_paquete)
    dependencias = raiz / "requirements-colab.txt"
    marca = Path("/content/aiweek_dependencias.sha256")
    version_deps = hash_archivo(dependencias)
    if not marca.exists() or marca.read_text() != version_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        marca.write_text(version_deps)
    for nombre in ("numpy", "pandas", "transformers", "tokenizers"):
        if nombre in sys.modules and getattr(sys.modules[nombre], "__version__", None) != importlib.metadata.version(nombre):
            raise RuntimeError("Reiniciar la sesión de Colab y ejecutar desde el principio para cargar las versiones instaladas")
    RESULTADOS = DRIVE / "resultados" / huella_paquete[:12]
else:
    raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/modelos.json").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("Abrir el notebook desde la carpeta del proyecto o notebooks")
    RESULTADOS = raiz / "data/experimentos_oficiales"
RAIZ = raiz
RESULTADOS.mkdir(parents=True, exist_ok=True)
if EN_COLAB:
    corpus_drive = RESULTADOS / "corpus_procesado.zip"
    marca_corpus = raiz / ".corpus_restaurado"
    if corpus_drive.is_file():
        huella_corpus = hash_archivo(corpus_drive)
        if not marca_corpus.exists() or marca_corpus.read_text() != huella_corpus:
            carpeta_corpus = raiz / "data/processed/corpus"
            with zipfile.ZipFile(corpus_drive) as z:
                for nombre in z.namelist():
                    if not (carpeta_corpus / nombre).resolve().is_relative_to(carpeta_corpus.resolve()):
                        raise ValueError("Ruta inválida en el corpus guardado")
                z.extractall(carpeta_corpus)
            marca_corpus.write_text(huella_corpus)
os.chdir(raiz)
sys.path.insert(0, str(raiz))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(raiz / "data/cache_huggingface") if EN_COLAB else os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Proyecto:", raiz)
print("Resultados:", RESULTADOS)

In [ ]:
import gc
import time
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scripts.auxiliares.oficial import (cargar_muestra, cargar_citaciones, metricas_recuperacion,
                                      consulta, guardar_json, huellas, abrir_experimento)
from scripts.auxiliares.recuperacion import (Encoder, BM25, construir_indice, cargar_corpus,
                                           leer_jsonl, hash_json, sha256)
from scripts.auxiliares.entorno import hardware

catalogo = json.loads((raiz / "configs/modelos.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
precision = "float16" if device == "cuda" else "float32"
batch_size = 32 if device == "cuda" and torch.cuda.get_device_properties(0).total_memory >= 30 * 2**30 else 4
print(hardware() if device == "cuda" else "CPU")
print("PyTorch:", torch.__version__, "Lote:", batch_size)

In [ ]:
modelos = ["multilingual-e5-large", "bge-m3", "jina-embeddings-v3", "multilingual-e5-base"]
tamanos = [320]
solapamiento = 32
reanudar = ""

muestra, entradas = cargar_muestra(raiz)
citas = cargar_citaciones(raiz)
docs = {d["doc_id"]: d for d in leer_jsonl(raiz / "data/processed/corpus/documentos.jsonl")}
configuracion = {"modelos": modelos, "tamanos": tamanos, "solapamiento": solapamiento,
                 "batch_size": batch_size, "precision": precision, "device": device,
                 "hardware": hardware() if device == "cuda" else "cpu", "torch": torch.__version__,
                 "huellas": huellas(raiz)}
carpeta = abrir_experimento(RESULTADOS, "encoders", configuracion, reanudar)
display(pd.DataFrame(catalogo["encoders"])[["nombre", "licencia", "recomendado", "revision"]])
print("Ejecución:", carpeta.name)

## Medición

El tamaño inicial es 320 tokens. Se puede ampliar tamanos para comparar otras ventanas en una ejecución nueva. Jina usa una licencia no comercial, tal como indica su ficha.

Se guarda cada combinación terminada. Para continuar después de una desconexión, copiar el nombre de la ejecución en reanudar. Los índices se comprimen en Drive y se pueden reutilizar desde 02.

In [ ]:
filas = []
for ficha in sorted(catalogo["encoders"], key=lambda f: f["prioridad"]):
    if ficha["nombre"] not in modelos:
        continue
    pendientes = [t for t in tamanos if not (carpeta / f"{ficha['nombre']}_{t}_resumen.json").exists()]
    if pendientes:
        encoder = Encoder(ficha, device, precision)
    for tamano in tamanos:
        etiqueta = f"{ficha['nombre']}_{tamano}"
        guardado = carpeta / f"{etiqueta}_resumen.json"
        if guardado.exists():
            filas.extend(json.loads(guardado.read_text(encoding="utf-8")))
            continue
        print("Indexando", etiqueta, flush=True)
        indice_local = raiz / "data/index/oficial" / carpeta.name / etiqueta
        config = {"encoder": ficha, "corpus": "data/processed/corpus", "tamano_tokens": tamano,
                  "solapamiento_tokens": solapamiento, "device": device, "precision": precision,
                  "batch_size": batch_size, "hibrido": False, "salida": str(indice_local)}
        manifiesto, recuperador = construir_indice(config, encoder)
        resumenes = []
        for hibrido in [False, True]:
            metodo = "hibrido" if hibrido else "denso"
            recuperador.bm25 = BM25([v["texto_busqueda"] for v in recuperador.ventanas]) if hibrido else None
            medidas = []
            for pregunta, entrada in zip(muestra, entradas):
                inicio = time.perf_counter()
                pasajes = recuperador.buscar(consulta(entrada), k=10)
                segundos = time.perf_counter() - inicio
                medidas.append({**metricas_recuperacion(pregunta, pasajes, docs, citas), "segundos": segundos,
                                "unidad_ids": [p["unidad_id"] for p in pasajes]})
            detalle = pd.DataFrame(medidas)
            detalle.to_json(carpeta / f"{etiqueta}_{metodo}_preguntas.json", orient="records", indent=2)
            resumenes.append({"modelo": ficha["nombre"], "tamano": tamano, "metodo": metodo,
                              "preguntas_con_referencia": int(detalle.referencias.gt(0).sum()),
                              **{c: float(detalle[c].mean()) for c in ["recall_normas_1", "recall_normas_5", "recall_normas_10"]},
                              "mediana_s": float(detalle.segundos.median()), "indexacion_s": manifiesto["segundos"],
                              "vram_pico_mib": manifiesto["vram_pico_mib"],
                              "indice_zip": (carpeta.relative_to(RESULTADOS) / f"{etiqueta}.zip").as_posix(),
                              "indice_sha256": manifiesto["sha256_indice"]})
            print(etiqueta, metodo, resumenes[-1]["recall_normas_10"], flush=True)
        archivo = shutil.make_archive(str(indice_local) + "_traslado", "zip", indice_local)
        shutil.copy2(archivo, carpeta / f"{etiqueta}.zip")
        guardar_json(guardado, resumenes)
        filas.extend(resumenes)
        del recuperador
        gc.collect()
    if pendientes:
        del encoder
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
resultados = pd.DataFrame(filas)
if len(resultados) != len(modelos) * len(tamanos) * 2:
    raise RuntimeError("La comparación quedó incompleta")
guardar_json(carpeta / "resumen.json", filas)
guardar_json(RESULTADOS / "ultima_comparacion_encoders.json", {"carpeta": str(carpeta.relative_to(RESULTADOS))})

In [ ]:
display(resultados.sort_values(["recall_normas_10", "recall_normas_5", "mediana_s"], ascending=[False, False, True]))
grafica = resultados.assign(etiqueta=resultados.modelo + " / " + resultados.metodo + " / " + resultados.tamano.astype(str))
grafica.set_index("etiqueta")[["recall_normas_5", "recall_normas_10"]].plot.barh(figsize=(10, 6), xlim=(0, 1))
plt.xlabel("Coincidencia con las normas de referencia")
plt.tight_layout()
plt.show()

## Resultados y decisiones

Pendientes de esta ejecución. Revisar también los artículos recuperados en los casos fallidos. Una coincidencia de norma puede corresponder a un artículo insuficiente.

En 02 se indica el encoder elegido. E5-large permanece como punto de partida hasta revisar estas tablas. La A100 permite comparar con más memoria, pero sus tiempos no representan el computador local.